# Healthcare Operations - Feature Engineering & Patient Segmentation
**Author:** Ravikant Yadav, Lead Data Analyst  
**Target:** Constructing Clinical Indicators, Demographic Buckets, Financial Profit Margins, and Aggregated Metrics

---

## 1. Executive Objective
In healthcare analytics, raw database tables must be transformed into highly specialized predictive and diagnostic features before modeling or loading into BI dashboards. This notebook constructs rich operational, clinical, and financial indicators:
- **Clinical Stays:** Age bucket divisions, is-weekend-admission, wait-time tiers.
- **Procedural Features:** Total count and sum cost of treatment procedures.
- **Financial Performance:** Out-of-pocket patient ratios, net profit margins, cost-recovery indicators.
- **Consolidated Matrix:** Merging these elements to export a master file to `data/processed`.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path('..')
data_dir = PROJECT_ROOT / 'data' / 'processed'

print(f"Ingesting processed data from: {data_dir.resolve()}")

Ingesting processed data from: C:\Users\yadav\healthcare-operations-analytics\data\processed


## 2. Ingestion of Cleaned Datasets

In [2]:
admissions = pd.read_csv(data_dir / 'admissions.csv')
patients = pd.read_csv(data_dir / 'patients.csv')
billing = pd.read_csv(data_dir / 'billing.csv')
treatments = pd.read_csv(data_dir / 'treatments.csv')
surveys = pd.read_csv(data_dir / 'satisfaction_surveys.csv')

admissions['admission_date'] = pd.to_datetime(admissions['admission_date'])
patients['birth_date'] = pd.to_datetime(patients['birth_date'])
print("Datasets successfully ingested.")

Datasets successfully ingested.


## 3. Engineering Inpatient Stays & Demographic Indicators
We build employee/patient age metrics and classify admissions as weekend or weekday arrivals.

In [3]:
# 1. Calculate Patient Age during admission
merged_patient = admissions.merge(patients, on='patient_id', how='left')
admissions['patient_age'] = (admissions['admission_date'] - merged_patient['birth_date']).dt.days // 365

# Bucketing patient age
admissions['age_group'] = pd.cut(
    admissions['patient_age'],
    bins=[-1, 18, 45, 65, 120],
    labels=['Pediatric', 'Young Adult', 'Middle Aged', 'Geriatric']
)

# 2. Is Weekend Admission Indicator
admissions['is_weekend_admission'] = admissions['admission_date'].dt.weekday.isin([5, 6]).astype(int)

# 3. Triage Patient Wait Time Buckets
admissions['wait_time_tier'] = pd.cut(
    admissions['wait_minutes'],
    bins=[-1, 15, 45, 120, 1440],
    labels=['Low (<=15)', 'Medium (15-45)', 'High (45-120)', 'Critical (>120)']
)

print(admissions[['admission_id', 'patient_age', 'age_group', 'is_weekend_admission', 'wait_time_tier']].head())

   admission_id  patient_age    age_group  is_weekend_admission  \
0             1           33  Young Adult                     0   
1             2           59  Middle Aged                     1   
2             3           81    Geriatric                     0   
3             4           66    Geriatric                     0   
4             5           31  Young Adult                     0   

   wait_time_tier  
0  Medium (15-45)  
1   High (45-120)  
2      Low (<=15)  
3   High (45-120)  
4  Medium (15-45)  


## 4. Aggregating Treatment Procedures
We roll up procedure metrics from the treatments file to join onto our master admission matrix.

In [4]:
treatment_rollup = treatments.groupby('admission_id').agg(
    total_procedures=('procedure_code', 'count'),
    accumulated_treatment_cost=('treatment_cost', 'sum')
).reset_index()

print("Treatment Rollup Sample:")
print(treatment_rollup.head())

Treatment Rollup Sample:
   admission_id  total_procedures  accumulated_treatment_cost
0             1                 3                     3257.00
1             2                 4                     3253.75
2             3                 1                     1307.44
3             4                 3                     3104.43
4             5                 4                     3903.21


## 5. Engineering Financial Margins and Metrics
We build hospital financial margins: net profit, recovery ratios, and out-of-pocket patient percentage of total charges.

In [5]:
billing['net_profit'] = billing['charge_amount'] - billing['cost_amount']
billing['profit_margin_pct'] = (billing['net_profit'] / billing['charge_amount']).round(4) * 100
billing['patient_out_of_pocket_ratio'] = (billing['patient_paid'] / billing['charge_amount']).round(4) * 100

print("Billing Metrics Sample:")
print(billing[['admission_id', 'net_profit', 'profit_margin_pct', 'patient_out_of_pocket_ratio']].head())

Billing Metrics Sample:
   admission_id  net_profit  profit_margin_pct  patient_out_of_pocket_ratio
0             1     3722.39              32.00                        16.73
1             2     3111.62              33.00                        23.58
2             3      700.70              33.67                        15.14
3             4      562.08              21.20                        19.24
4             5      788.80              22.63                        16.01


## 6. Constructing Master Feature Matrix
We assemble these indicators into a single, comprehensive row-grain dataset representing all patient clinical-operational-financial cycles.

In [6]:
master_matrix = admissions.merge(treatment_rollup, on='admission_id', how='left') \
                             .merge(billing, on='admission_id', how='left') \
                             .merge(surveys, on='admission_id', how='left')

# Impute missing rolled up values
master_matrix['total_procedures'] = master_matrix['total_procedures'].fillna(0)
master_matrix['accumulated_treatment_cost'] = master_matrix['accumulated_treatment_cost'].fillna(0.0)

print("Consolidated Feature Matrix Shape:", master_matrix.shape)
print(master_matrix.info())

Consolidated Feature Matrix Shape: (45000, 32)
<class 'pandas.DataFrame'>
RangeIndex: 45000 entries, 0 to 44999
Data columns (total 32 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   admission_id                 45000 non-null  int64         
 1   patient_id_x                 45000 non-null  int64         
 2   doctor_id                    45000 non-null  int64         
 3   department_id_x              45000 non-null  int64         
 4   admission_date               45000 non-null  datetime64[us]
 5   discharge_date               45000 non-null  str           
 6   length_of_stay               45000 non-null  int64         
 7   admission_type               45000 non-null  str           
 8   wait_minutes                 45000 non-null  float64       
 9   severity_score               45000 non-null  float64       
 10  readmission_30d              45000 non-null  bool          
 11  disch

## 7. Saving Consolidated Output

In [7]:
master_matrix.to_csv(data_dir / 'consolidated_patient_features.csv', index=False)
print("Consolidated analytical data matrix exported successfully to data/processed!")

Consolidated analytical data matrix exported successfully to data/processed!
